<a href="https://colab.research.google.com/github/jimmyGit538/coin-market-cap-project/blob/production/Jessica/CMC_api_project_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#remove this for couldrun

from google.colab import auth
auth.authenticate_user()
print("Authenticated")


Authenticated


In [2]:

# Install and import libraries


!pip install --quiet requests pandas google-cloud-bigquery db-dtypes

import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from getpass import getpass

from google.cloud import bigquery
import time

# Configuration section


# GCP project
PROJECT_ID = "coinmarketcapproject"

# Dataset where new tables will live
DATASET_ID = "crypto_raw"

# destination tables
TABLE_MAP_NEW = "map_gcp_raw"
TABLE_CATEGORY_TOP20 = "category_top20_29_raw"
TABLE_CATEGORY_TOP20_COINS = "category_top20_coins_29_raw"



# v1 for map + category endpoint
BASE_URL_V1 = "https://pro-api.coinmarketcap.com/v1"



# prompts securely.
CMC_API_KEY = getpass("Enter your CoinMarketCap API key: ").strip()

# Common request headers for CMC
HEADERS = {
    "Accepts": "application/json",
    "X-CMC_PRO_API_KEY": CMC_API_KEY
}

# BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID)


Enter your CoinMarketCap API key: ··········


In [3]:
FOCUS_CATEGORY_IDS = [
    "604f274bebccdd50cd175fbb",
    "6051a82d66fc1b42617d6dd0",
    "692b02c1c0b341673d681a21",
    "6246aade491a5b4fe942fa3f",
    "6634dccba7b6f0637eec196a",
    "63248a04694d2a40b403f244",
    "604f2749ebccdd50cd175fb9",
    "6051a82666fc1b42617d6dc8",
    "5fb62da404d1dd4c73744883",
    "677d0fc06bd44718911d7781",
    "61213ce7c049672cb6ce7eb8",
    "604f2776ebccdd50cd175fdc",
    "6051a82366fc1b42617d6dc4",
    "692b0302c0b341673d681a27",
    "6053df006be1bf5c15e865ec",
    "604f2743ebccdd50cd175fb5",
    "6051a80866fc1b42617d6da1",
    "6051a81666fc1b42617d6db2",
    "6433de7df79a2653906cd680",
    "65f23191e6c934565751ce16",
    "67250af2622a021a2592cba5",
    "604f2753ebccdd50cd175fc1",
    "68638d58358e0763b448b3ca",
    "604f273debccdd50cd175fb0",
    "60291fa0db1be76c46298e83",
    "6051a81a66fc1b42617d6db7",
    "6051a82166fc1b42617d6dc1",
    "5fb62883c9ddcc213ed13308",
    "6051a82566fc1b42617d6dc6"
]

category_ids_to_pull = FOCUS_CATEGORY_IDS
print("Categories to pull:", len(category_ids_to_pull))


Categories to pull: 29


In [4]:
# Helper functions

def call_cmc(endpoint, params=None, version="v1"):
    base = BASE_URL_V1 if version == "v1" else BASE_URL_V3
    url = f"{base}/{endpoint}"
    r = requests.get(url, headers=HEADERS, params=params or {}, timeout=60)
    r.raise_for_status()
    return r.json().get("data", {})

def write_df_to_bigquery(df, table_name, write_disposition="WRITE_APPEND"):
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    job_config = bigquery.LoadJobConfig(
        write_disposition=write_disposition,
        autodetect=True
    )
    load_job = bq_client.load_table_from_dataframe(df, table_id, job_config=job_config)
    load_job.result()
    print(f"Loaded {len(df)} rows into {table_id}")



In [5]:
import time
from requests.exceptions import HTTPError


In [6]:
# MAP endpoint


def pull_all_map(listing_status="active", sort="cmc_rank", page_size=5000):
    """
    Pulls all rows from the CoinMarketCap cryptocurrency map endpoint
    using pagination.

    listing_status: for example "active"
    sort: for example "cmc_rank"
    page_size: number of rows per request, CMC usually allows up to 5000
    """
    all_rows = []
    start = 1

    while True:
        print(f"Pulling map data starting at {start}")

        params = {
            "listing_status": listing_status,
            "sort": sort,
            "start": start,
            "limit": page_size
        }

        data_chunk = call_cmc(
            endpoint="cryptocurrency/map",
            params=params
        )

        if not data_chunk:
            print("No more data returned. Stopping.")
            break

        all_rows.extend(data_chunk)

        # If we got less than the page size, that means we reached the end
        if len(data_chunk) < page_size:
            print("Last page received. Stopping.")
            break

        # Move the start pointer forward for the next request
        start += page_size

    return all_rows


# Call the map endpoint and get all rows
map_data = pull_all_map(
    listing_status="active",
    sort="cmc_rank",
    page_size=5000
)

# Convert to DataFrame
df_map = pd.DataFrame(map_data)

# Add ingestion timestamp
df_map["ingestion_timestamp_utc"] = pd.Timestamp.utcnow()

# Write to BigQuery
write_df_to_bigquery(
    df_map,
    TABLE_MAP_NEW,
    write_disposition="WRITE_APPEND"
)



Pulling map data starting at 1
Pulling map data starting at 5001
Pulling map data starting at 10001
Last page received. Stopping.
Loaded 10022 rows into coinmarketcapproject.crypto_raw.map_gcp_raw


In [7]:
#Run ONE loop that builds BOTH tables

from pandas import json_normalize

category_frames = []
coin_rows = []

for cat_id in category_ids_to_pull:
    print(f"Pulling category detail + coins for {cat_id}")

    cat_data = call_cmc(
        endpoint="cryptocurrency/category",
        params={"id": cat_id},
        version="v1"
    )

    # Category detail table
    df_cat = json_normalize(cat_data)
    df_cat["source_category_id"] = str(cat_id)
    df_cat["ingestion_timestamp_utc"] = pd.Timestamp.utcnow()
    category_frames.append(df_cat)

    # Coins table
    coins = cat_data.get("coins", []) or []
    for coin in coins:
        usd = (coin.get("quote") or {}).get("USD", {}) or {}
        coin_rows.append({
            "category_id": str(cat_id),
            "coin_id": coin.get("id"),
            "coin_name": coin.get("name"),
            "coin_symbol": coin.get("symbol"),
            "coin_slug": coin.get("slug"),
            "coin_cmc_rank": coin.get("cmc_rank"),
            "coin_market_cap": usd.get("market_cap"),
            "coin_price": usd.get("price"),
            "coin_volume_24h": usd.get("volume_24h"),
            "ingestion_timestamp_utc": pd.Timestamp.utcnow(),
        })

    print(f"  coins pulled: {len(coins)}")
    time.sleep(2)

df_category_top = pd.concat(category_frames, ignore_index=True)
if "coins" in df_category_top.columns:
    df_category_top["coins"] = df_category_top["coins"].astype(str)

df_category_coins = pd.DataFrame(coin_rows)

print("Category detail rows:", len(df_category_top))
print("Category coin rows:", len(df_category_coins))
print("Unique coins:", df_category_coins["coin_symbol"].nunique())

df_category_coins.head()



Pulling category detail + coins for 604f274bebccdd50cd175fbb
  coins pulled: 15
Pulling category detail + coins for 6051a82d66fc1b42617d6dd0
  coins pulled: 28
Pulling category detail + coins for 692b02c1c0b341673d681a21
  coins pulled: 41
Pulling category detail + coins for 6246aade491a5b4fe942fa3f
  coins pulled: 35
Pulling category detail + coins for 6634dccba7b6f0637eec196a
  coins pulled: 33
Pulling category detail + coins for 63248a04694d2a40b403f244
  coins pulled: 57
Pulling category detail + coins for 604f2749ebccdd50cd175fb9
  coins pulled: 58
Pulling category detail + coins for 6051a82666fc1b42617d6dc8
  coins pulled: 58
Pulling category detail + coins for 5fb62da404d1dd4c73744883
  coins pulled: 60
Pulling category detail + coins for 677d0fc06bd44718911d7781
  coins pulled: 72
Pulling category detail + coins for 61213ce7c049672cb6ce7eb8
  coins pulled: 64
Pulling category detail + coins for 604f2776ebccdd50cd175fdc
  coins pulled: 64
Pulling category detail + coins for 6051

,category_id,coin_id,coin_name,coin_symbol,coin_slug,coin_cmc_rank,coin_market_cap,coin_price,coin_volume_24h,ingestion_timestamp_utc
0,604f274bebccdd50cd175fbb,1437,Zcash,ZEC,zcash,17.0,6.867334e+09,417.422723,5.340917e+08,2025-12-23 22:56:27.644350+00:00
1,604f274bebccdd50cd175fbb,22691,Starknet,STRK,starknet-token,98.0,3.961138e+08,0.079786,4.313397e+07,2025-12-23 22:56:27.644368+00:00
2,604f274bebccdd50cd175fbb,1712,Quantum Resistant Ledger,QRL,quantum-resistant-ledger,223.0,1.555406e+08,2.289478,2.473378e+05,2025-12-23 22:56:27.644376+00:00
3,604f274bebccdd50cd175fbb,4948,Nervos Network,CKB,nervos-network,242.0,1.137885e+08,0.002376,3.489890e+06,2025-12-23 22:56:27.644382+00:00
4,604f274bebccdd50cd175fbb,5858,QANplatform,QANX,qanplatform,609.0,2.725848e+07,0.015350,3.327824e+05,2025-12-23 22:56:27.644387+00:00


In [8]:
#Write to BigQuery (one-time reset so only the 29 exist)

write_df_to_bigquery(df_category_top, TABLE_CATEGORY_TOP20, "WRITE_APPEND")
write_df_to_bigquery(df_category_coins, TABLE_CATEGORY_TOP20_COINS, "WRITE_APPEND")


Loaded 29 rows into coinmarketcapproject.crypto_raw.category_top20_29_raw
Loaded 2170 rows into coinmarketcapproject.crypto_raw.category_top20_coins_29_raw
